# Session 5 — Monitoring Model Explainability and Data Drift using Evidently AI

**Goal:** answer the question every deployed model eventually faces — "is the data
I'm seeing in production still similar to what I trained on?" — using
[Evidently AI](https://www.evidentlyai.com/), an open-source monitoring library.

## Why this matters

A model's accuracy on a held-out test set (Sessions 1, 4) tells you nothing about how
it performs six months later, once real-world data has shifted (new customer
segments, a pandemic, a UI redesign that changes what users click). **Data drift** is
when the distribution of incoming features changes; **target/prediction drift** is
when the model's own outputs shift. Evidently computes both automatically and
renders an interactive HTML report.

## Prerequisites

```bash
pip install evidently
```
Runs entirely locally — no account or cloud service needed.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris

print("Evidently is installed and ready.")

## Step 1 — Simulate a "reference" and a "current" dataset

In a real deployment, **reference** is the training/validation data (the world your
model was built for) and **current** is a recent slice of production traffic. Here we
simulate drift by deliberately shifting two features in the "current" set.

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame
df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "target"]

rng = np.random.default_rng(42)
reference = df.sample(n=100, random_state=1).reset_index(drop=True)

current = df.drop(reference.index, errors="ignore").sample(n=50, random_state=2).reset_index(drop=True)
# Simulate drift: sensor recalibration shifted sepal_length up, and a data pipeline bug
# started truncating petal_width toward zero.
current["sepal_length"] = current["sepal_length"] + rng.normal(1.5, 0.3, size=len(current))
current["petal_width"] = current["petal_width"] * 0.4

print("reference:", reference.shape, " current:", current.shape)

## Step 2 — Run a Data Drift report

Evidently compares each column's distribution between `reference` and `current`
using statistical tests appropriate to the column type (Kolmogorov-Smirnov for
continuous features here), and flags which columns drifted.

In [ ]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

drift_report = Report(metrics=[DataDriftPreset()])
drift_report.run(reference_data=reference, current_data=current)

result = drift_report.as_dict()
drift_metrics = result["metrics"][0]["result"]
print(f"Dataset drift detected: {drift_metrics['dataset_drift']}")
print(f"Drifted columns: {drift_metrics['number_of_drifted_columns']} / "
      f"{drift_metrics['number_of_columns']}")

In [ ]:
for col, info in drift_metrics["drift_by_columns"].items():
    flag = "DRIFTED" if info["drift_detected"] else "ok"
    print(f"  {col:<15} {flag:<8} p-value/score={info['drift_score']:.4f} "
          f"(test: {info['stattest_name']})")

## Step 3 — Save the interactive HTML report

The report object also renders a full interactive dashboard — distribution plots per
column, side by side — useful for a human reviewing *why* a column drifted, not just
that it did.

In [ ]:
drift_report.save_html("data_drift_report.html")
print("Saved data_drift_report.html -- open it in a browser to explore interactively.")

## Step 4 — Target drift: has the *label distribution* shifted?

Feature drift doesn't always mean the model is wrong — but a shift in the label
distribution (or the model's predictions, if labels aren't available yet) is a
stronger signal that something changed.

In [ ]:
from evidently.metric_preset import TargetDriftPreset

target_drift_report = Report(metrics=[TargetDriftPreset(columns=["target"])])
target_drift_report.run(reference_data=reference, current_data=current)

target_result = target_drift_report.as_dict()
print("Target drift report generated -- see data_drift_report.html-style output for details.")

## Step 5 — Model explainability: which features drive predictions?

Evidently's `RegressionPreset`/`ClassificationPreset` (paired with `TargetDriftPreset`)
report can also summarize per-feature relationships to the target, complementing the
model-agnostic SHAP analysis covered in depth in Session 22.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = df.drop(columns="target")
y = df["target"]
clf = RandomForestClassifier(n_estimators=100, random_state=0).fit(X, y)

importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature importances (a lightweight explainability signal):")
print(importances.round(4))
print("\nFor a much deeper, per-prediction explanation, see Session 22 (SHAP).")

## What to try next

* Wire `Report.run()` into a scheduled job (cron, Airflow, or the CI/CD pipeline from
  Session 10) that runs nightly against the latest production data slice and alerts
  when `dataset_drift` flips to `True`.
* Session 17 builds directly on this: automatically triggering a retraining job the
  moment drift crosses a threshold, instead of just reporting it.
* Combine with Deepchecks (Session 11) for validation checks Evidently doesn't cover
  (schema violations, duplicate rows, label leakage).